# Let's estimate how much seasonal snow our dataset doesn't capture. 

Since S1 IW mode VV data isn't captured over Greenland, the Canadian Arctic Archipelago, and the Russian Arctic Islands, let's add up the areas with seasonal snow across these. We'll use the Sturm & Liston 2021 dataset so we don't count area with ice. We don't include Antarctica here because most sources show Antarctic seasonal snow extent as negligible and the Sturm & Liston 2021 dataset seems to have some strange artifacting in Antarctica.  

**Run this notebook BEFORE `calculate_spatial_coverage_and_temporal_resolution.ipynb`** — that one reads the `seasonal_snow_excluded_area_summary.csv` written here to build Table 1.

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import xyzservices as xyz
import cartopy
from pyproj import Transformer
import cartopy.crs as ccrs
from cartopy import feature as cfeature
from global_snowmelt_runoff_onset.config import Config, Tile
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import geopandas as gpd
import numpy as np
import zarr
import rioxarray
import rasterio
import tqdm
import pandas as pd
import pickle
import os
from pathlib import Path

In [ ]:
# v10: the v9 phenology store (snowmelt/snow_cover/global_modis_snow_cover.zarr) no longer
# exists, so this notebook cannot run on the v9 config at all. The excluded-area numbers
# themselves are version-independent — the phenology dataset is only used for its CRS, as
# the reprojection target for the Sturm & Liston raster — so v10 reproduces the v9 totals.
config = Config('config/global_config_v10.txt')

# Results are scoped by dataset version so a v10 run cannot overwrite the v9 tables.
# Switching versions = editing the config path above.
VERSION = config.version
RESULTS_DIR = Path('results') / VERSION
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
modis_seasonal_snow_mask_ds = xr.open_zarr(config.snow_phenology_store, decode_coords="all")
modis_seasonal_snow_mask_ds

In [ ]:
world_gdf = gpd.read_file(
    "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_countries.zip"
)
greenland_gdf = world_gdf[world_gdf['ADMIN'] == 'Greenland']
greenland_proj_gdf = greenland_gdf.to_crs(modis_seasonal_snow_mask_ds.rio.crs)
greenland_proj_gdf['geometry'] = greenland_proj_gdf.simplify(100)

In [ ]:
# read canada_poly.zip in data folder, which is a shapefile of the canadian arctic archipelago
canadian_archipelago_gdf = gpd.read_file("data/canada_poly.zip")
canadian_archipelago_proj_gdf = canadian_archipelago_gdf.to_crs(modis_seasonal_snow_mask_ds.rio.crs)
canadian_archipelago_proj_gdf['geometry'] = canadian_archipelago_proj_gdf.simplify(100)

In [ ]:
russia_gdf = world_gdf[world_gdf['ADMIN'] == 'Russia']
russia_proj_gdf = russia_gdf.to_crs(modis_seasonal_snow_mask_ds.rio.crs)
# filter out all but the largest polygon by area, you can access like geom.area in russia_proj_gdf.geometry.iloc[0].geoms to get the individual polygons, and then calculate area for each and filter out all but the largest one. The largest one is the mainland, the second largest one is the russian arctic archipelago, and the rest are small islands that we can ignore for this analysis.
russia_proj_gdf = russia_proj_gdf.explode(index_parts=False)
russia_proj_gdf['area'] = russia_proj_gdf.geometry.area
russia_proj_gdf = russia_proj_gdf.sort_values('area', ascending=False).reset_index(drop=True)
russia_proj_gdf['number'] = russia_proj_gdf.index
# remove 0 and 1, which are the mainland and the russian arctic archipelago, respectively
russia_proj_gdf = russia_proj_gdf[russia_proj_gdf['number'] > 1]
# remove all below latitude of 70 N, first convert 70N to y location using transform, not cartopy but pyproj
transformer = Transformer.from_crs("EPSG:4326", modis_seasonal_snow_mask_ds.rio.crs, always_xy=True)
y_coord = transformer.transform(0, 70)
russian_archipelago_proj_gdf = russia_proj_gdf[russia_proj_gdf.geometry.centroid.y > y_coord[1]]
russian_archipelago_proj_gdf['geometry'] = russian_archipelago_proj_gdf.simplify(100)

In [ ]:
f,ax=plt.subplots()
greenland_proj_gdf.plot(ax=ax, color='blue')
canadian_archipelago_proj_gdf.plot(ax=ax, color='green')
russian_archipelago_proj_gdf.plot(ax=ax, color='red')

In [ ]:
snow_classification_da = rioxarray.open_rasterio(
    "data/SnowClass_GL_01km_30.0arcsec_2021_v01.0.tif",
    chunks=True,
    mask_and_scale=False,
).squeeze()
snow_classification_da

In [ ]:
greenland_snow_classification_da = snow_classification_da.rio.clip_box(*greenland_gdf.total_bounds)
greenland_snow_classification_proj_da = greenland_snow_classification_da.rio.reproject(modis_seasonal_snow_mask_ds.rio.crs, resampling=rasterio.enums.Resampling.mode)
greenland_snow_classification_proj_da = greenland_snow_classification_proj_da.rio.clip(
    greenland_proj_gdf.geometry,
    drop=True,
    invert=False,
)
# values of 1,2,3,5,6 correspond to snow classes. create binary mask where 1,2,3,5,6 are 1 and the rest are 0
greenland_snow_binary_da = xr.where(
    (greenland_snow_classification_proj_da.isin([1,2,3,5,6])), 
    1, 
    0
)
pixel_count = greenland_snow_binary_da.sum().values
pixel_area_km2 = (greenland_snow_binary_da.rio.resolution()[0] / 1000) ** 2
greenland_snow_area_km2 = pixel_count * pixel_area_km2
greenland_snow_area_km2
# 363,919 km^2

In [ ]:
canadian_archipelago_snow_classification_da = snow_classification_da.rio.clip_box(*canadian_archipelago_gdf.total_bounds)
canadian_archipelago_snow_classification_proj_da = canadian_archipelago_snow_classification_da.rio.reproject(modis_seasonal_snow_mask_ds.rio.crs, resampling=rasterio.enums.Resampling.mode)
canadian_archipelago_snow_classification_proj_da = canadian_archipelago_snow_classification_proj_da.rio.clip(
    canadian_archipelago_proj_gdf.geometry,
    drop=True,
    invert=False,
)
# values of 1,2,3,5,6 correspond to snow classes. create binary mask where 1,2,3,5,6 are 1 and the rest are 0
canadian_archipelago_snow_binary_da = xr.where(
    (canadian_archipelago_snow_classification_proj_da.isin([1,2,3,5,6])), 
    1, 
    0
)
pixel_count = canadian_archipelago_snow_binary_da.sum().values
pixel_area_km2 = (canadian_archipelago_snow_binary_da.rio.resolution()[0] / 1000) ** 2
canadian_archipelago_snow_area_km2 = pixel_count * pixel_area_km2
canadian_archipelago_snow_area_km2
# 1,235,852 km^2

In [ ]:
# Split into western and eastern hemispheres to avoid anti-meridian issue
russian_4326_bounds = russian_archipelago_proj_gdf.to_crs("EPSG:4326").total_bounds

# Western part (positive longitudes, e.g., 140°E to 180°E)
west_da = snow_classification_da.rio.clip_box(minx=140, miny=russian_4326_bounds[1], maxx=180, maxy=russian_4326_bounds[3])
west_proj_da = west_da.rio.reproject(modis_seasonal_snow_mask_ds.rio.crs, resampling=rasterio.enums.Resampling.mode)
west_clipped_da = west_proj_da.rio.clip(russian_archipelago_proj_gdf.geometry, drop=True, invert=False)
west_binary_da = xr.where(west_clipped_da.isin([1, 2, 3, 5, 6]), 1, 0)
west_pixel_count = west_binary_da.sum().values

# Eastern part (negative longitudes, e.g., -180°W to -140°W)  
east_da = snow_classification_da.rio.clip_box(minx=-180, miny=russian_4326_bounds[1], maxx=-140, maxy=russian_4326_bounds[3])
east_proj_da = east_da.rio.reproject(modis_seasonal_snow_mask_ds.rio.crs, resampling=rasterio.enums.Resampling.mode)
east_clipped_da = east_proj_da.rio.clip(russian_archipelago_proj_gdf.geometry, drop=True, invert=False)
east_binary_da = xr.where(east_clipped_da.isin([1, 2, 3, 5, 6]), 1, 0)
east_pixel_count = east_binary_da.sum().values

# Combine results
total_pixel_count = west_pixel_count + east_pixel_count
pixel_area_km2 = (modis_seasonal_snow_mask_ds.rio.resolution()[0] / 1000) ** 2
russian_archipelago_snow_area_km2 = total_pixel_count * pixel_area_km2
russian_archipelago_snow_area_km2
# 9,080 km^2

In [ ]:
# could also add in the seasonal snow over the ice-free areas of antartica too...... upper bound would probably be 54,274 km2
# there is no good estimate of seasonal snow over the ice-free areas of antarctica, so the best we can do is
# use the total ice-free area of antarctica as an upper bound, which is 54,274 km2. This is not a good estimate of seasonal snow over antarctica, but it is the best we can do with the data we have.
# From Brooks et al 2019: "We calculated the current total ice-free area of Antarctica to be 0.44% (54,274 km2)...."
antarctica_excluded_area_km2 = 54274

In [ ]:
total_excluded_snow_area = greenland_snow_area_km2 + canadian_archipelago_snow_area_km2 + russian_archipelago_snow_area_km2 + antarctica_excluded_area_km2
total_excluded_snow_area
# pre antarctica: 1,608,851 km^2
# with antarctica: 1,663,126 km^2

In [ ]:
results_df = pd.DataFrame({
    'Region': ['Greenland', 'Canadian Arctic Archipelago', 'Russian Arctic Archipelago', 'Antarctica', 'Total Excluded Area'],
    'Snow Area (km^2)': [greenland_snow_area_km2, canadian_archipelago_snow_area_km2, russian_archipelago_snow_area_km2, antarctica_excluded_area_km2, total_excluded_snow_area]
})
results_df

In [ ]:
# This total excluded area of 1.6 million km^2 is quoted in the section 3.3 of the paper... 
# "The spatial coverage of our global snowmelt runoff onset dataset is constrained by the availability of Sentinel-1 Interferometric Wide mode data. 
# Sentinel-1 acquires data over Antarctica and much of the Arctic in Extra Wide mode or IW HH/HV, so our dataset does not include ~1.6 million km2 of seasonal snow over 
# the ice-free areas of Antarctica, Greenland, the Canadian Arctic Archipelago, and the Russian Arctic Islands (S. T. Brooks et al., 2019; Liston & Sturm, 2021)."
results_df.to_csv(RESULTS_DIR / "seasonal_snow_excluded_area_summary.csv", index=False)